# Accuracy Improvement Playbook (Per Run)

This notebook defines what each experiment run should deliver, how to diagnose weak runs, and what target metrics we should aim for.


## 1) Per-run checklist (must-have outputs)

For every run, capture and keep these fields (already available in leaderboard/reward leaderboard):

- Config: `seed`, `timesteps`, `learning_rate`, `gamma`, `ent_coef`, reward scales, threshold/horizon
- Validation quality: `val_overall_accuracy`, `val_actionable_accuracy`, `val_trade_win_rate`
- Test quality: `test_overall_accuracy`, `test_actionable_accuracy`, `test_trade_win_rate`
- Signal return: `val_cumulative_signal_return`, `test_cumulative_signal_return`
- Reward behavior: `val_reward_total_mean`, `val_reward_direction_mean`, `val_reward_drawdown_mean`

A run is considered *promotion-ready* only if both validation and test are healthy (not just one split).


## 2) Target bands to aim for

These are practical near-term targets for this project stage:

| Metric | Minimum acceptable | Strong target | Stretch target |
|---|---:|---:|---:|
| Validation actionable accuracy | 0.50 | 0.55 | 0.60+ |
| Test actionable accuracy | 0.50 | 0.55 | 0.60+ |
| Validation trade win rate | 0.50 | 0.55 | 0.60+ |
| Test trade win rate | 0.50 | 0.55 | 0.60+ |
| Validation cumulative signal return | > 0.00 | > 0.05 | > 0.10 |
| Test cumulative signal return | > 0.00 | > 0.03 | > 0.08 |
| Mean reward total (validation) | > 0.00 | clearly positive | stable positive |

Notes:
- Prioritize `actionable_accuracy` and `trade_win_rate` over `overall_accuracy` because Hold-heavy models can inflate overall accuracy.
- Require at least 3 seeds before declaring a configuration robust.


In [ ]:
from pathlib import Path
import pandas as pd

leaderboard_path = Path('..') / 'data' / 'experiment_leaderboard.csv'
reward_path = Path('..') / 'data' / 'experiment_reward_leaderboard.csv'

leaderboard = pd.read_csv(leaderboard_path) if leaderboard_path.exists() else pd.DataFrame()
reward_leaderboard = pd.read_csv(reward_path) if reward_path.exists() else pd.DataFrame()

print('leaderboard rows:', len(leaderboard))
print('reward leaderboard rows:', len(reward_leaderboard))
leaderboard.head(3)


In [ ]:
targets = {
    'val_actionable_accuracy': 0.55,
    'test_actionable_accuracy': 0.55,
    'val_trade_win_rate': 0.55,
    'test_trade_win_rate': 0.55,
    'val_cumulative_signal_return': 0.05,
    'test_cumulative_signal_return': 0.03,
    'val_reward_total_mean': 0.0,
}

if leaderboard.empty:
    print('No experiment leaderboard found yet.')
else:
    best = leaderboard.iloc[0]
    rows = []
    for metric, target in targets.items():
        if metric not in best.index:
            rows.append({'metric': metric, 'current': None, 'target': target, 'gap': None})
            continue
        current = float(best[metric])
        rows.append({
            'metric': metric,
            'current': current,
            'target': target,
            'gap': current - target,
        })
    gap_df = pd.DataFrame(rows).sort_values('gap')
    display(gap_df)


## 3) Tuning guidance by failure mode

Use this diagnosis after each run batch:

- **Low actionable accuracy + low win rate**
  - Increase `timesteps` and test lower `learning_rate`.
  - Increase `reward_direction_scale` slightly.

- **Good accuracy but poor cumulative return**
  - Increase `reward_return_scale` and `reward_drawdown_penalty_scale`.
  - Re-check horizon/threshold alignment with trading objective.

- **Too many Holds / weak actionability**
  - Reduce hold penalty conservatively only if noise trading appears.
  - Sweep `ent_coef` to encourage exploration when policy is too conservative.

- **Validation good, test weak (overfit)**
  - Favor simpler settings and wider seed coverage.
  - Promote configs that remain stable across at least 3 seeds.


In [ ]:
def run_grade(row):
    checks = {
        'val_actionable_accuracy': row.get('val_actionable_accuracy', 0.0) >= 0.55,
        'test_actionable_accuracy': row.get('test_actionable_accuracy', 0.0) >= 0.55,
        'val_trade_win_rate': row.get('val_trade_win_rate', 0.0) >= 0.55,
        'test_trade_win_rate': row.get('test_trade_win_rate', 0.0) >= 0.55,
        'val_cumulative_signal_return': row.get('val_cumulative_signal_return', -1.0) > 0.0,
        'test_cumulative_signal_return': row.get('test_cumulative_signal_return', -1.0) > 0.0,
    }
    score = sum(checks.values())
    if score >= 6:
        grade = 'A (promotion-ready)'
    elif score >= 4:
        grade = 'B (promising)'
    elif score >= 2:
        grade = 'C (needs tuning)'
    else:
        grade = 'D (rework)'
    return grade, checks

if not leaderboard.empty:
    best = leaderboard.iloc[0].to_dict()
    grade, checks = run_grade(best)
    print('Best run grade:', grade)
    print('Checks:')
    for k, v in checks.items():
        print(f'  {k}:', 'PASS' if v else 'FAIL')


## 4) Standard run protocol

1. Run a batch with at least 3 seeds.
2. Sort by `ranking_score` and inspect top 5.
3. Re-rank top 5 by reward stability (`val_reward_total_mean`, drawdown penalty behavior).
4. Promote only runs that pass both accuracy and return thresholds on validation and test.
5. Archive winning config as next baseline before trying more aggressive settings.
